# Notebook 04: Preprocessing

**Purpose:** Apply preprocessing pipeline to all images (grayscale, deskew, adaptive binarization, normalize).

In [ ]:
import sys
sys.path.append('./docuspend/src')

from pathlib import Path
import yaml
import json
import os
from preprocessing import ReceiptPreprocessor
from tqdm import tqdm

base_dir = Path("/content/docuspend") if os.path.exists("/content") else Path("./docuspend")
print(f"Working with base directory: {base_dir}")

In [ ]:
# Step 1: Load preprocessing config
with open(base_dir / "configs/preprocessing_config.yaml", 'r') as f:
    config = yaml.safe_load(f)

print("Preprocessing Configuration:")
for key, value in config.get('preprocessing', {}).items():
    if isinstance(value, dict):
        print(f"  {key}: {list(value.keys())}")
    else:
        print(f"  {key}: {value}")

In [ ]:
# Step 2 & 3: Batch preprocessing loop
preprocessor = ReceiptPreprocessor(config)

splits = ["train", "val", "test"]
for split in splits:
    input_dir = base_dir / f"data/raw/{split}"
    output_dir = base_dir / f"data/processed/{split}"
    output_dir.mkdir(parents=True, exist_ok=True)
    
    image_files = sorted(input_dir.glob("*.jpg"))
    print(f"\nProcessing {split}: {len(image_files)} images...")
    
    for image_file in tqdm(image_files):
        output_file = output_dir / image_file.name.replace(".jpg", ".png")
        preprocessor.preprocess_receipt(str(image_file), str(output_file))

print(f"\n✓ Preprocessing complete")

In [ ]:
# Step 4: Generate visualizations (before/after)
import cv2
import matplotlib.pyplot as plt
import random

random.seed(42)

fig, axes = plt.subplots(15, 2, figsize=(12, 30))
plot_idx = 0

for split in splits:
    raw_dir = base_dir / f"data/raw/{split}"
    processed_dir = base_dir / f"data/processed/{split}"
    
    image_files = list(raw_dir.glob("*.jpg"))
    sample_files = random.sample(image_files, min(5, len(image_files)))
    
    for image_file in sample_files:
        if plot_idx >= 15:
            break
        
        # Original
        original = cv2.imread(str(image_file), cv2.IMREAD_GRAYSCALE)
        axes[plot_idx, 0].imshow(original, cmap='gray')
        axes[plot_idx, 0].set_title(f"Original: {image_file.name}")
        axes[plot_idx, 0].axis('off')
        
        # Preprocessed
        processed_file = processed_dir / image_file.name.replace(".jpg", ".png")
        if processed_file.exists():
            processed = cv2.imread(str(processed_file), cv2.IMREAD_GRAYSCALE)
            axes[plot_idx, 1].imshow(processed, cmap='gray')
            axes[plot_idx, 1].set_title(f"Preprocessed: {processed_file.name}")
        
        axes[plot_idx, 1].axis('off')
        plot_idx += 1

plt.tight_layout()
viz_dir = base_dir / "outputs/visualizations"
viz_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(viz_dir / "preprocessing_samples.png", dpi=100, bbox_inches='tight')
print(f"✓ Saved visualization to {viz_dir / 'preprocessing_samples.png'}")
plt.close()

In [ ]:
# Step 5: Generate preprocessing statistics
stats = preprocessor.get_statistics()

stats_output = {
    "total_processed": stats["total_processed"],
    "successful": stats["successful"],
    "failed": stats["failed"],
    "average_time_seconds": round(stats["average_time"], 2)
}

logs_dir = base_dir / "outputs/logs"
logs_dir.mkdir(parents=True, exist_ok=True)
with open(logs_dir / "preprocessing_stats.json", 'w') as f:
    json.dump(stats_output, f, indent=2)

print("Preprocessing Statistics:")
for k, v in stats_output.items():
    print(f"  {k}: {v}")

In [ ]:
# Step 6: Output status report
print("="*60)
print("PREPROCESSING COMPLETE")
print("="*60)
print(f"✅ Processed {stats['total_processed']} images")
print(f"✅ Successful: {stats['successful']}")
print(f"✅ Failed: {stats['failed']}")
print(f"✅ Average time: {stats['average_time']:.2f}s per image")
print(f"✅ Output saved to data/processed/")
print(f"\nNext: Run 05_fine_tuning.ipynb")